In [1]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import seaborn as sns
from matplotlib.image import imread
from PIL import Image
import tensorflow as tf
np.random.seed(1337)
import gc

import glob

from tensorflow.keras import layers
from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import Input, Activation,Conv2DTranspose, Dropout, AveragePooling2D,Flatten, Dense, Conv2D,MaxPool2D, MaxPooling2D, BatchNormalization,Conv2DTranspose,concatenate,UpSampling2D
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')


img_size = 128
print(os.listdir())
dataset = os.listdir("music_seperation_dataset/train")
labels = dataset
print(labels)

['.git', '.vscode', 'drum_part.wav', 'drum_prediction.wav', 'full.wav', 'mask.png', 'model_classes_3.ipynb', 'model_classes_full.ipynb', 'model_separation.ipynb', 'music_dataset', 'music_dataset_spectro_3_instrument', 'music_dataset_spectro_full', 'music_seperation_dataset', 'my_model_3.keras', 'my_model_full.keras', 'my_separator_model_full.keras', 'README.md', 'separated_part.png', 'spectrogramMaker.py']
['Acoustic_Guitar', 'Bass_Guitar', 'Drum_set', 'Electric_Guitar', 'full_mix', 'Keyboard']


In [2]:
def get_dataset_array(data_dir):
    data = []
    path = os.path.join(data_dir)
    for img in os.listdir(data_dir):
            try:
                img_arr = cv2.imread(os.path.join(path,img))
                resized_arr = cv2.resize(img_arr, (img_size, img_size))
                data.append([resized_arr])
                gc.collect()
            except Exception as e:
                print(e)
    return np.array(data,dtype="float32") 

In [ ]:

x_train = get_dataset_array("music_seperation_dataset/train/full_mix")
y_train = get_dataset_array("music_seperation_dataset/train/Drum_set")

x_test = get_dataset_array("music_seperation_dataset/test/full_mix")
y_test = get_dataset_array("music_seperation_dataset/test/Drum_set")

x_valid = get_dataset_array("music_seperation_dataset/valid/full_mix")
y_valid = get_dataset_array("music_seperation_dataset/valid/Drum_set")



In [4]:
gc.collect()
x_train = np.array(x_train)/255
gc.collect()
x_test = np.array(x_test)/255
gc.collect()
x_valid = np.array(x_valid)/255
gc.collect()
y_train = np.array(y_train)/255
gc.collect()
y_test = np.array(y_test)/255
gc.collect()
y_valid = np.array(y_valid)/255
gc.collect()

0

In [20]:
x_train = x_train.reshape(-1, img_size, img_size, 3)
y_train = y_train.reshape(-1, img_size, img_size, 3)

x_valid = x_valid.reshape(-1, img_size, img_size, 3)
y_valid = y_valid.reshape(-1, img_size, img_size, 3)

x_test = x_test.reshape(-1, img_size, img_size, 3)
y_test = y_test.reshape(-1, img_size, img_size, 3)

print(x_train.dtype)
print(x_train.shape)
print(y_train.shape)

float32
(48, 128, 128, 3)
(48, 128, 128, 3)


In [ ]:
num_classes = 1
def Unet():
    inputs =  layers.Input(shape=(None,None,1))

    conv1 = Conv2D(16, (3,3), activation = 'relu', padding='same')(inputs)
    conv1 = Conv2D(16, (3,3), activation = 'relu', padding='same')(conv1)
    pool1 = MaxPool2D((2,2))(conv1)

    conv2 = Conv2D(32, (3,3), activation = 'relu', padding='same')(pool1)
    conv2 = Conv2D(32, (3,3), activation = 'relu', padding='same')(conv2)
    pool2 = MaxPool2D((2,2))(conv2)

    conv3 = Conv2D(64, (3,3), activation = 'relu', padding='same')(pool2)
    conv3 = Conv2D(64, (3,3), activation = 'relu', padding='same')(conv3)
    pool3 = MaxPool2D((2,2))(conv3)

    conv4 = Conv2D(128, (3,3), activation = 'relu', padding='same')(pool3)
    conv4 = Conv2D(128, (3,3), activation = 'relu', padding='same')(conv4)
    pool4 = MaxPool2D((2,2))(conv4)

    conv5 = Conv2D(256, (3,3), activation = 'relu', padding='same')(pool4)
    conv5 = Conv2D(256, (3,3), activation = 'relu', padding='same')(conv5)

    goUp1 = Conv2DTranspose(128,(2,2),strides=(2,2),padding='same',activation = 'relu',)(conv5)
    goUp1 = concatenate([goUp1,conv4])
    conv6 = Conv2D(128, (3,3), activation = 'relu', padding='same')(goUp1)
    conv6 = Conv2D(128, (3,3), activation = 'relu', padding='same')(conv6)

    goUp2 = Conv2DTranspose(64,(2,2),strides=(2,2),padding='same',activation = 'relu',)(conv6)
    goUp2 = concatenate([goUp2,conv3])
    conv7 = Conv2D(64, (3,3), activation = 'relu', padding='same')(goUp2)
    conv7 = Conv2D(64, (3,3), activation = 'relu', padding='same')(conv7)

    goUp3 = Conv2DTranspose(32,(2,2),strides=(2,2),padding='same',activation = 'relu')(conv7)
    goUp3 = concatenate([goUp3,conv2])
    conv8 = Conv2D(32, (3,3), activation = 'relu', padding='same')(goUp3)
    conv8 = Conv2D(32, (3,3), activation = 'relu', padding='same',)(conv8)

    goUp4 = Conv2DTranspose(16,(2,2),strides=(2,2),padding='same',activation = 'relu')(conv8)
    goUp4 = concatenate([goUp4,conv1])
    conv9 = Conv2D(16, (3,3), activation = 'relu', padding='same')(goUp4)
    conv9 = Conv2D(16, (3,3), activation = 'relu', padding='same')(conv9)



    outputs = Conv2D(num_classes,(1,1),activation="sigmoid")(conv9)

    model = Model(inputs=[inputs],outputs=[outputs])
    return model

model = Unet()
model.compile(
              optimizer = 'adam', loss = 'mse',
              metrics = ['mae']
              )
     

In [37]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_38 (Conv2D)  │ (None, None,      │        448 │ input_layer_2[0]… │
│                     │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_39 (Conv2D)  │ (None, None,      │      2,320 │ conv2d_38[0][0]   │
│                     │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_8     │ (None, None,      │          0 │ conv2d_39[0][0]   │
│ (MaxPooling2D)      │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_40 (Conv2D)  │ (None, None,      │      4,640 │ max_pooling2d_8[… │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_41 (Conv2D)  │ (None, None,      │      9,248 │ conv2d_40[0][0]   │
│                     │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_9     │ (None, None,      │          0 │ conv2d_41[0][0]   │
│ (MaxPooling2D)      │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_42 (Conv2D)  │ (None, None,      │     18,496 │ max_pooling2d_9[… │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_43 (Conv2D)  │ (None, None,      │     36,928 │ conv2d_42[0][0]   │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_10    │ (None, None,      │          0 │ conv2d_43[0][0]   │
│ (MaxPooling2D)      │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_44 (Conv2D)  │ (None, None,      │     73,856 │ max_pooling2d_10… │
│                     │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_45 (Conv2D)  │ (None, None,      │    147,584 │ conv2d_44[0][0]   │
│                     │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_11    │ (None, None,      │          0 │ conv2d_45[0][0]   │
│ (MaxPooling2D)      │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_46 (Conv2D)  │ (None, None,      │    295,168 │ max_pooling2d_11… │
│                     │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_47 (Conv2D)  │ (None, None,      │    590,080 │ conv2d_46[0][0]   │
│                     │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_8  │ (None, None,      │    131,200 │ conv2d_47[0][0]   │
│ (Conv2DTranspose)   │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_8       │ (None, None,      │          0 │ conv2d_transpose

 Total params: 1,941,105 (7.40 MB)

 Trainable params: 1,941,105 (7.40 MB)

 Non-trainable params: 0 (0.00 B)

In [38]:
batch_size = 5
n_epochs = 100
model.fit(x_train, y_train, batch_size = batch_size,
                    epochs = n_epochs, validation_data = (x_valid, y_valid))

Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 105ms/step - loss: 0.1565 - mae: 0.3526 - val_loss: 0.0312 - val_mae: 0.1370
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - loss: 0.1484 - mae: 0.3414 - val_loss: 0.0376 - val_mae: 0.1471
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - loss: 0.1485 - mae: 0.3292 - val_loss: 0.0351 - val_mae: 0.1477
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - loss: 0.1172 - mae: 0.3027 - val_loss: 0.0487 - val_mae: 0.1657
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - loss: 0.0935 - mae: 0.2498 - val_loss: 0.1191 - val_mae: 0.2775
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - loss: 0.0535 - mae: 0.1639 - val_loss: 0.1484 - val_mae: 0.2984
Epoch 7/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - loss: 0.0458 - mae: 0.1459 - val_loss: 0.1522 - val_mae: 0.3024
Epoch 8/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - loss: 0.0406 - mae: 0.1361 - val_loss: 0.1621 - val_mae: 0.3177
Epoch 9/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - l

In [24]:
gc.collect()

model.save('my_separator_model_full.keras')

In [ ]:
img = cv2.imread("music_seperation_dataset/test/full_mix/53_full_mix.png")

img=cv2.resize(img,(128,128))
img= np.reshape(img,(-1, 128, 128, 3))
img=img/255

predimg= np.squeeze(model.predict(img))

prediction= predimg*255
print(prediction.shape)



cv2.imwrite("separated_part.png",prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
(128, 128)


True

: 

In [28]:
import librosa

image = cv2.imread("separated_part.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=1000,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('drum_prediction.wav',audio, 22050)




In [ ]:
image = cv2.imread("music_seperation_dataset/test/full_mix/53_full_mix.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=64,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('full.wav',audio, 22050)

In [ ]:
image = cv2.imread("music_seperation_dataset/test/Drum_set/53_Drum_set.png",0)

to_db = (image.astype(np.float32)/255)*80-80

power = librosa.db_to_power(to_db)

audio = librosa.feature.inverse.mel_to_audio(power,n_fft = 2048, hop_length = 512,n_iter=1000,sr=22050)


audio = librosa.util.normalize(audio)

import soundfile
soundfile.write('drum_part.wav',audio, 22050)